In [ ]:
!nvidia-smi

Sun May 31 19:15:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!pip install -q huggingface_hub

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="AY000554/Car_plate_detecting_dataset",
    repo_type="dataset",
    local_dir="/content/car_plate_dataset"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

'/content/car_plate_dataset'

проверка готовности датасета

In [3]:
!find /content/car_plate_dataset -maxdepth 3 -type d

/content/car_plate_dataset
/content/car_plate_dataset/resources
/content/car_plate_dataset/resources/images
/content/car_plate_dataset/.cache
/content/car_plate_dataset/.cache/huggingface
/content/car_plate_dataset/.cache/huggingface/download


In [4]:
!find /content/car_plate_dataset -maxdepth 3 -type f | head -50

/content/car_plate_dataset/license.txt
/content/car_plate_dataset/resources/images/avto-nomera-02.vv139e.jpg
/content/car_plate_dataset/.cache/huggingface/CACHEDIR.TAG
/content/car_plate_dataset/.cache/huggingface/.gitignore
/content/car_plate_dataset/README.md
/content/car_plate_dataset/test.zip
/content/car_plate_dataset/train.zip
/content/car_plate_dataset/.gitattributes
/content/car_plate_dataset/val.zip


распаковка архивов

In [5]:
!unzip -q /content/car_plate_dataset/train.zip -d /content/car_plate_dataset/train
!unzip -q /content/car_plate_dataset/val.zip -d /content/car_plate_dataset/val
!unzip -q /content/car_plate_dataset/test.zip -d /content/car_plate_dataset/test

проверка

In [6]:
!find /content/car_plate_dataset -maxdepth 3 -type d

/content/car_plate_dataset
/content/car_plate_dataset/resources
/content/car_plate_dataset/resources/images
/content/car_plate_dataset/.cache
/content/car_plate_dataset/.cache/huggingface
/content/car_plate_dataset/.cache/huggingface/download
/content/car_plate_dataset/val
/content/car_plate_dataset/val/val
/content/car_plate_dataset/val/val/images
/content/car_plate_dataset/val/val/labels
/content/car_plate_dataset/train
/content/car_plate_dataset/train/train
/content/car_plate_dataset/train/train/images
/content/car_plate_dataset/train/train/labels
/content/car_plate_dataset/test
/content/car_plate_dataset/test/test
/content/car_plate_dataset/test/test/images
/content/car_plate_dataset/test/test/labels


создаем data.yaml

In [ ]:
%%writefile /content/car_plate_dataset/data.yaml

path: /content/car_plate_dataset

train: train/train/images
val: val/val/images
test: test/test/images

names:
  0: license_plate

Writing /content/car_plate_dataset/data.yaml


проверка что файл создался

In [ ]:
!cat /content/car_plate_dataset/data.yaml


path: /content/car_plate_dataset

train: train/train/images
val: val/val/images
test: test/test/images

names:
  0: license_plate


In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.1 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
print("YOLO ready")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO ready


подключим гугл драйв

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Исследование влияния архитектуры и гиперпараметров

## Обучение YOLOv8n

### обучение YOLOv8n (imgsz=640)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/car_plate_dataset/data.yaml",
    epochs=10,
    imgsz=640,
    batch=16,
    name="plate_yolov8n_640"
)

тестовая выборка

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/plate_yolov8n_640/weights/best.pt")

metrics = model.val(
    data="/content/car_plate_dataset/data.yaml",
    split="test",
    imgsz=640,
    name="test_yolov8n_640"
)

print(metrics.results_dict)

сохранение в гугл драйв

In [ ]:
!mkdir -p /content/drive/MyDrive/plate_detection_practice/yolov8n_640

!cp /content/runs/detect/plate_yolov8n_640/weights/best.pt /content/drive/MyDrive/plate_detection_practice/yolov8n_640/best.pt
!cp /content/runs/detect/plate_yolov8n_640/results.csv /content/drive/MyDrive/plate_detection_practice/yolov8n_640/results.csv
!cp -r /content/runs/detect/test_yolov8n_640 /content/drive/MyDrive/plate_detection_practice/yolov8n_640/test_results

In [ ]:
!ls /content/drive/MyDrive/plate_detection_practice/yolov8n_640

### обучение YOLOv8n (imgsz=416)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/car_plate_dataset/data.yaml",
    epochs=10,
    imgsz=416,
    batch=16,
    name="plate_yolov8n_416"
)

проверим на тест выборке

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/plate_yolov8n_416/weights/best.pt")

metrics_416 = model.val(
    data="/content/car_plate_dataset/data.yaml",
    split="test",
    imgsz=416,
    name="test_yolov8n_416"
)

print(metrics_416.results_dict)

сохранение в гугл драйв

In [ ]:
!mkdir -p /content/drive/MyDrive/plate_detection_practice/yolov8n_416

!cp /content/runs/detect/plate_yolov8n_416/weights/best.pt /content/drive/MyDrive/plate_detection_practice/yolov8n_416/best.pt
!cp /content/runs/detect/plate_yolov8n_416/results.csv /content/drive/MyDrive/plate_detection_practice/yolov8n_416/results.csv
!cp -r /content/runs/detect/test_yolov8n_416 /content/drive/MyDrive/plate_detection_practice/yolov8n_416/test_results

In [ ]:
!ls /content/drive/MyDrive/plate_detection_practice/yolov8n_416

## Обучение YOLOv8s

### обучение YOLOv8s (epochs=10, imgsz=640)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="/content/car_plate_dataset/data.yaml",
    epochs=10,
    imgsz=640,
    batch=16,
    name="plate_yolov8s_640_10"
)

тестовая выборка

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/plate_yolov8s_640_10/weights/best.pt")

metrics_s_640_10 = model.val(
    data="/content/car_plate_dataset/data.yaml",
    split="test",
    imgsz=640,
    name="test_yolov8s_640_10"
)

print(metrics_s_640_10.results_dict)

сохранение в гугл драйв

In [ ]:
!mkdir -p /content/drive/MyDrive/plate_detection_practice/yolov8s_640_10

!cp /content/runs/detect/plate_yolov8s_640_10/weights/best.pt /content/drive/MyDrive/plate_detection_practice/yolov8s_640_10/best.pt
!cp /content/runs/detect/plate_yolov8s_640_10/results.csv /content/drive/MyDrive/plate_detection_practice/yolov8s_640_10/results.csv
!cp -r /content/runs/detect/test_yolov8s_640_10 /content/drive/MyDrive/plate_detection_practice/yolov8s_640_10/test_results

In [ ]:
!ls /content/drive/MyDrive/plate_detection_practice/yolov8s_640_10

### обучение YOLOv8s (epochs=10, imgsz=416)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="/content/car_plate_dataset/data.yaml",
    epochs=10,
    imgsz=416,
    batch=16,
    name="plate_yolov8s_416_10"
)

тестовая выборка

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/plate_yolov8s_416_10/weights/best.pt")

metrics_s_416_10 = model.val(
    data="/content/car_plate_dataset/data.yaml",
    split="test",
    imgsz=416,
    name="test_yolov8s_416_10"
)

print(metrics_s_416_10.results_dict)

сохранить на диск

In [ ]:
!mkdir -p /content/drive/MyDrive/plate_detection_practice/yolov8s_416_10

!cp /content/runs/detect/plate_yolov8s_416_10/weights/best.pt /content/drive/MyDrive/plate_detection_practice/yolov8s_416_10/best.pt
!cp /content/runs/detect/plate_yolov8s_416_10/results.csv /content/drive/MyDrive/plate_detection_practice/yolov8s_416_10/results.csv
!cp -r /content/runs/detect/test_yolov8s_416_10 /content/drive/MyDrive/plate_detection_practice/yolov8s_416_10/test_results

In [ ]:
!ls /content/drive/MyDrive/plate_detection_practice/yolov8s_416_10

### обучение YOLOv8s (epochs=20, imgsz=640)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="/content/car_plate_dataset/data.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    name="plate_yolov8s_640_20"
)

тестовая выборка

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/plate_yolov8s_640_20/weights/best.pt")

metrics_s_640_20 = model.val(
    data="/content/car_plate_dataset/data.yaml",
    split="test",
    imgsz=640,
    name="test_yolov8s_640_20"
)

print(metrics_s_640_20.results_dict)

сохранение в гугл драйв

In [ ]:
!mkdir -p /content/drive/MyDrive/plate_detection_practice/yolov8s_640_20

!cp /content/runs/detect/plate_yolov8s_640_20/weights/best.pt /content/drive/MyDrive/plate_detection_practice/yolov8s_640_20/best.pt
!cp /content/runs/detect/plate_yolov8s_640_20/results.csv /content/drive/MyDrive/plate_detection_practice/yolov8s_640_20/results.csv
!cp -r /content/runs/detect/test_yolov8s_640_20 /content/drive/MyDrive/plate_detection_practice/yolov8s_640_20/test_results

### обучение YOLOv8s (epochs=20, imgsz=416)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="/content/car_plate_dataset/data.yaml",
    epochs=20,
    imgsz=416,
    batch=16,
    name="plate_yolov8s_416_20"
)

тестовая выборка

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/plate_yolov8s_416_20/weights/best.pt")

metrics_s_416_20 = model.val(
    data="/content/car_plate_dataset/data.yaml",
    split="test",
    imgsz=416,
    name="test_yolov8s_416_20"
)

print(metrics_s_416_20.results_dict)

сохраним в гугл драйв

In [ ]:
!mkdir -p /content/drive/MyDrive/plate_detection_practice/yolov8s_416_20

!cp /content/runs/detect/plate_yolov8s_416_20/weights/best.pt /content/drive/MyDrive/plate_detection_practice/yolov8s_416_20/best.pt
!cp /content/runs/detect/plate_yolov8s_416_20/results.csv /content/drive/MyDrive/plate_detection_practice/yolov8s_416_20/results.csv
!cp -r /content/runs/detect/test_yolov8s_416_20 /content/drive/MyDrive/plate_detection_practice/yolov8s_416_20/test_results

# Оптимизация разрядности вычислений (квантование)

## YOLOv8s (epochs=20, imgsz=416)

In [ ]:
from ultralytics import YOLO

Загружаем веса с гугл драйва

In [ ]:
model_path = "/content/drive/MyDrive/plate_detection_practice/yolov8s_416_20/best.pt"
model = YOLO(model_path)

Запуск теста в FP32 (стандартная точность)

In [ ]:
metrics_fp32 = model.val(
    data="/content/car_plate_dataset/data.yaml",
    split="test",
    imgsz=416,
    half=False,
    name="test_yolov8s_416_fp32"
)

print(metrics_fp32.results_dict)

Запуск теста в FP16

In [ ]:
metrics_fp16 = model.val(
    data="/content/car_plate_dataset/data.yaml",
    split="test",
    imgsz=416,
    half=True,
    name="test_yolov8s_416_fp16"
)

print(metrics_fp16.results_dict)

In [ ]:
!cp -r /content/runs/detect/test_yolov8s_416_fp32 /content/drive/MyDrive/plate_detection_practice/yolov8s_416_20/
!cp -r /content/runs/detect/test_yolov8s_416_fp16 /content/drive/MyDrive/plate_detection_practice/yolov8s_416_20/

## YOLOv8s (epochs=20, imgsz=416)

In [ ]:
from ultralytics import YOLO

In [ ]:
model = YOLO("/content/drive/MyDrive/plate_detection_practice/yolov8s_640_20/best.pt")

Запуск теста в FP32 (стандартная точность)

In [ ]:
metrics_fp32 = model.val(
    data="/content/car_plate_dataset/data.yaml",
    split="test",
    imgsz=640,
    half=False,
    name="test_yolov8s_640_fp32"
)

Запуск теста в FP16

In [ ]:
metrics_fp16 = model.val(
    data="/content/car_plate_dataset/data.yaml",
    split="test",
    imgsz=640,
    half=True,
    name="test_yolov8s_640_fp16"
)

In [ ]:
!cp -r /content/runs/detect/test_yolov8s_640_fp32 /content/drive/MyDrive/plate_detection_practice/yolov8s_640_20/
!cp -r /content/runs/detect/test_yolov8s_640_fp16 /content/drive/MyDrive/plate_detection_practice/yolov8s_640_20/

# Смена среды исполнения (экспорт в формат ONNX)

## YOLOv8s (epochs=20, imgsz=640)

In [ ]:
from ultralytics import YOLO

In [ ]:
model = YOLO("/content/drive/MyDrive/plate_detection_practice/yolov8s_640_20/best.pt")

Экспортируем в ONNX

In [ ]:
exported_path = model.export(format="onnx")

print(f"Успешно экспортировано в ONNX! Путь: {exported_path}")

In [ ]:
model_onnx = YOLO("/content/drive/MyDrive/plate_detection_practice/yolov8s_640_20/best.onnx")

metrics_onnx = model_onnx.val(
    data="/content/car_plate_dataset/data.yaml",
    split="test",
    imgsz=640,
    name="test_yolov8s_640_onnx"
)

сохранение в гугл драйв

In [ ]:
!cp -r /content/runs/detect/test_yolov8s_640_onnx /content/drive/MyDrive/plate_detection_practice/yolov8s_640_20/

## YOLOv8s (epochs=20, imgsz=416)

экспорт модели в ONNX

In [ ]:
from ultralytics import YOLO

In [ ]:
model_416 = YOLO("/content/drive/MyDrive/plate_detection_practice/yolov8s_416_20/best.pt")

exported_path_416 = model_416.export(format="onnx")

print(f"Успешно экспортировано в ONNX! Путь: {exported_path_416}")

In [ ]:
model_onnx_416 = YOLO("/content/drive/MyDrive/plate_detection_practice/yolov8s_416_20/best.onnx")

metrics_onnx_416 = model_onnx_416.val(
    data="/content/car_plate_dataset/data.yaml",
    split="test",
    imgsz=416,
    name="test_yolov8s_416_onnx"
)

In [ ]:
!cp -r /content/runs/detect/test_yolov8s_416_onnx /content/drive/MyDrive/plate_detection_practice/yolov8s_416_20/